In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_8.pickle

In [ ]:
%%RecordEvent
%%time
### cell 9 ###

# Standardize column names for consistency
col_rename_map_2022 = {
    'Country': 'In which country do you currently reside?',
    'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'
}
responses_df_2022.rename(columns=col_rename_map_2022, inplace=True)

col_rename_map_2017 = {
    'Country': 'In which country do you currently reside?',
    'CurrentJobTitleSelect': 'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
}
responses_df_2017.rename(columns=col_rename_map_2017, inplace=True)

# Apply value mappings
responses_df_2017['In which country do you currently reside?'] = (
    responses_df_2017['In which country do you currently reside?']
    .replace({
        'United States': 'United States of America',
        "People 's Republic of China": 'China',
        'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland'
    })
)
responses_df_2022['In which country do you currently reside?'] = (
    responses_df_2022['In which country do you currently reside?']
    .replace({'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'})
)

# Consolidate into subset and label others
for df in [responses_df_2017, responses_df_2018, responses_df_2019, responses_df_2020, responses_df_2021, responses_df_2022]:
    df['In which country do you currently reside?'] = (
        df['In which country do you currently reside?']
        .where(df['In which country do you currently reside?'].isin(subset_of_countries), 'Other')
    )

# Optimized combine function
def combine_subset_of_data_from_multiple_years_18(question_of_interest, x_axis_title, include_2017=None):
    dfs_years = [
        (responses_df_2018, '2018'),
        (responses_df_2019, '2019'),
        (responses_df_2020, '2020'),
        (responses_df_2021, '2021'),
        (responses_df_2022, '2022')
    ]
    if include_2017:
        dfs_years.insert(0, (responses_df_2017, '2017'))

    combined = []
    for df, year in dfs_years:
        total = df[question_of_interest].count()
        vc = df[question_of_interest].value_counts(dropna=False).sort_index()
        pct = (vc * 100 / total).round(1).to_frame('percentage')
        pct['year'] = year
        pct[x_axis_title] = pct.index
        combined.append(pct)

    df_combined = pd.concat(combined)
    df_combined.columns = ['percentage', 'year', x_axis_title]
    return df_combined

# Produce the final dataframe and chart
country_df_combined = (
    combine_subset_of_data_from_multiple_years_18(
        question_of_interest, title_for_x_axis, include_2017=True
    )
    .sort_values(['year', 'percentage'])
)

country_df_combined = (
    country_df_combined
    .rename(columns={'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom (UK)'})
    .replace({'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom'})
)

bar_chart_multiple_years_18(
    country_df_combined,
    title_for_x_axis,
    column_of_interest,
    title_for_chart,
    title_for_y_axis
)

print("'Other' = any country not shown")

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_9_try_8.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_9_try_8.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[9], f)


In [ ]:
opt_output = Out.get(4)